# **Aprendizaje Automático 1 - Trabajo Práctico 2**
## EDA - Predicción de lluvias en Australia

**Integrantes:** Alejandro Acosta, Diego Hauret, Gastón Rinaldi, Agustín Torres

Análisis exploratorio de datos previo a la construcción de un modelo de ...

**Indice:**
1. Carga y comprensión de los datos
2. Análisis de valores faltantes
3. Imputación de valores faltantes (ejemplo)
4. Análisis univariado (variable objetivo y predictoras)
5. Análisis bivariado (correlaciones y relación con MEDV)
6. Multicolinealidad entre predictoras
7. Detección de outliers
8. Conclusiones y próximos pasos


## Diccionario de variables

| Variable | Descripción |
| :--- | :--- |
| **Unnamed** | Índices |
| **Date** | Fecha |
| **Location** | Localidad de ubicación de la estación meteorológica |
| **MinTemp** | Temperatura mínima en °C |
| **MaxTemp** | Temperatura máxima en °C |
| **Rainfall** | Cantidad de lluvia registrada durante las últimas 24 horas (mm) |
| **Evaporation** | Evaporación aparente calculada (medida en mm) |
| **Sunshine** | Número de horas totales de sol brillante registradas a lo largo del día |
| **WindGustDir** | Dirección de la ráfaga de viento más fuerte |
| **WindGustSpeed** | Velocidad de la ráfaga de viento más fuerte |
| **WindDir9am** | Dirección del viento registrada a las 09:00 horas |
| **WindDir3pm** | Dirección del viento registrada a las 15:00 horas |
| **WindSpeed9am** | Velocidad del viento (en km/h). Promedio de los 10 minutos anteriores |
| **WindSpeed3pm** | Velocidad del viento (en km/h). Promedio de los 10 minutos anteriores |
| **Humidity9am** | Porcentaje de humedad relativa registrado a las 09:00 horas |
| **Humidity3pm** | Porcentaje de humedad relativa registrado a las 15:00 horas |
| **Pressure9am** | Presión atmosférica a las 09:00 horas (en hectopascales) |
| **Pressure3pm** | Presión atmosférica a las 15:00 horas (en hectopascales) |
| **Cloud9am** | Nubosidad a las 09:00 horas (0 indica cielo despejado y 8 nublado total) |
| **Cloud3pm** | Nubosidad a las 15:00 horas (0 indica cielo despejado y 8 nublado total) |
| **Temp9am** | Temperatura medida a las 09:00 horas en °C |
| **Temp3pm** | Temperatura medida a las 15:00 horas en °C |
| **RainToday** | Variable categórica binaria (Yes / No). Toma el valor Yes si la precipitación acumulada (Rainfall) en las últimas 24 horas hasta las 09:00 horas de día actual superó el 1.0 mm |
| **RainTomorrow** | Variable objetivo (target) a predecir (Yes / No). Indica si lloverá al día siguiente (las precipitaciones de mañana superarán el umbral de 1.0 mm) |

## 0. Carga de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
"""
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, average_precision_score,
                             matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                             precision_recall_curve, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

#sns.set_theme(style="whitegrid")
"""

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Carga y comprensión de los datos

In [ ]:
df = pd.read_csv('./weatherAUS_2026C2.csv')
#df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Aprendizaje_automatico/weatherAUS_2026C2.csv')
print("Dimensiones:", df.shape)
df.head(10)


In [ ]:
df.info()

## 2. Observaciones sobre las columnas del archivo


* `Unnamed: 0`: es un índice que no aporta información, entonces se elimina.
* `RainfallTomorrow`: es la **cantidad de lluvia (mm) de mañana**. Es una fuga de información (*data leakage*): RainTomorrow se define justamente como "mañana llovió ≥ 1 mm", por lo que esta columna casi determina el target. En un caso real **no la conoceríamos al momento de predecir**. Se elimina de los predictores.

In [ ]:
df = df.drop(columns=["Unnamed: 0"])
df = df.sort_values(["Date"]).reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"])

num_cols = df.select_dtypes(include="number").columns.drop("RainfallTomorrow").tolist()
cat_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]

df = df.drop(columns=["RainfallTomorrow"])
print("Período:", df["Date"].min().date(), "->", df["Date"].max().date())
print("Ubicaciones de las estaciones:", df["Location"].nunique())
print("Filas duplicadas:", df.duplicated().sum())
print("\nVariables numéricas:", num_cols)
print("Variables categóricas:", cat_cols)

In [ ]:
df.info()

## 3. Análisis de valores faltantes en columna objetivo


Revisamos cuántos nulos tiene cada columna y si hay algún patrón (filas con muchos nulos a la vez, o nulos concentrados).


In [ ]:
# Se eliminan las filas sin target o sin RainToday (se hace ANTES de dividir: no usa información de otras filas)
n0 = len(df)
df = df.dropna(subset=["RainTomorrow", "RainToday"]).copy()
print(f"Filas eliminadas: {n0 - len(df)} ({(n0 - len(df)) / n0:.2%}) | Filas restantes: {len(df)}")

## 4. Implementación de un modelo de AutoML con Pycaret

In [ ]:
# 1. Importar el módulo de clasificación de PyCaret
from pycaret.classification import *

# A. IMPORTANTE: Ordenamos el dataframe por la columna de fecha ya que se trata de
# una serie de tiempo y PyCaret necesita que los datos estén ordenados cronológicamente
# para poder hacer la división Train/Test correctamente.
df = df.sort_values(by='Date').reset_index(drop=True)

# B. Configurar PyCaret indicando la estructura de Serie de Tiempo
# PyCaret imputará nulos, codificará variables categóricas y dividirá los datos (Train/Test)
exp_lluvia = setup(
    data=df,
    target='RainTomorrow',
    session_id=RANDOM_STATE,
    data_split_shuffle=False,
    data_split_stratify=False,  # Desactiva la estratificación
    fold_strategy='timeseries',
    fold=5,
    fix_imbalance=True,
    normalize=True
)

# 3. Entrenar y comparar todos los modelos
# PyCaret probará algoritmos como XGBoost, LightGBM, Random Forest, etc.
# Te mostrará una tabla ordenada por la mejor métrica (Accuracy, AUC, Recall...)
print("Entrenando modelos...")
mejor_modelo = compare_models()

# 4. Ajustar el modelo (Tuning de hiperparámetros)
# Optimiza el algoritmo ganador para extraer su máximo rendimiento
print("Optimizando modelos...")
modelo_optimizado = tune_model(mejor_modelo)

# 5. Evaluar el modelo (Gráficas e interpretabilidad)
# Abre una interfaz para ver la Matriz de Confusión, Curva ROC e Importancia de Variables
print("Evaluando mejor modelo...")
evaluate_model(modelo_optimizado)

# 6. Finalizar el modelo
# Entrena el algoritmo optimizado utilizando el 100% de los datos de tu dataframe df
print("Evaluando modelo final...")
modelo_final = finalize_model(modelo_optimizado)

# 7. Guardar el pipeline completo
# Genera un archivo 'pipeline_lluvia.pkl' con todo el preprocesamiento y el modelo listo
print("Guardando modelo...")
save_model(modelo_final, 'pipeline_lluvia')
